# Closet: tagging accuracy test (Phase 2)

Compares **FashionCLIP** and **Marqo-FashionSigLIP** on your own hand-checked clothing photos and
recommends which one your Mac should use (`EMBED_MODEL`). Targets: **category ≥ 90%, type ≥ 70%**.

**Before you start**
1. On your Mac: `cd worker && uv run closet-worker export-eval` → writes `~/ClosetPhotos/eval/`.
2. Open `labels.csv` there and fix any wrong rows (category, type, pattern, color). Aim for ~50 rows.
   Types should use names from `vocab/types.yaml` (e.g. `oxford shirt`, `chinos`, `kurta`).
3. Run the export again so the zip has your corrected labels, then come back here.

**Runtime → Change runtime type → T4 GPU** (free), then run the cells top to bottom.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: it still works, just slower."

## 1. Upload `closet-eval.zip`

In [ ]:
import pathlib, shutil, zipfile
from google.colab import files

uploaded = files.upload()  # pick ~/ClosetPhotos/eval/closet-eval.zip
name = next(iter(uploaded))
root = pathlib.Path("/content/eval")
shutil.rmtree(root, ignore_errors=True)
with zipfile.ZipFile(name) as z:
    z.extractall(root)
print(len(list((root / "photos").glob("*.jpg"))), "photos,", sum(1 for _ in open(root / "labels.csv")) - 1, "label rows")

## 2. Install the libraries (≈1 minute)

In [ ]:
!pip -q install open_clip_torch "rembg[gpu]" scikit-image scikit-learn pyyaml

## 3. Run both models
First run downloads the models (~1.5 GB). Background removal runs once and is shared by both models.

In [ ]:
import sys
sys.path.insert(0, str(root / "code"))
from eval.tagging import run, print_table
from closet_worker.vision.embed import MODELS

result = run(root, MODELS, cutout=True, device="cuda")
print_table(result)

## 4. Where it goes wrong

In [ ]:
import pandas as pd
from IPython.display import Image, display

labels = pd.read_csv(root / "labels.csv").fillna("")
for model, r in result["models"].items():
    p = pd.DataFrame(r["predictions"]).add_prefix("pred_").rename(columns={"pred_file": "file"})
    m = labels.merge(p, on="file")
    wrong = m[(m["type"].str.lower() != m["pred_type"].str.lower()) | (m["category"] != m["pred_category"])]
    print(f"\n{model}: {len(wrong)} of {len(m)} rows wrong on type or category")
    display(wrong[["file", "item_name", "category", "pred_category", "type", "pred_type"]])
    png = root / f"confusion_{model.replace('/', '_')}.png"
    if png.exists():
        display(Image(str(png)))

## 5. Download the report
Then on your Mac set `EMBED_MODEL=<recommended>` in `worker/.env`. If the winner is Marqo-FashionSigLIP (768-d embeddings), tell Claude so it can add the migration that resizes the embedding column.

In [ ]:
files.download(str(root / "report.json"))
print("Recommended EMBED_MODEL =", result["recommended"])